# TEMPAL_SOPHIAS — Kaggle-Ready Implementation Notebook

**Project:** TEMPAL: Temporal-Perceptual Multimodal Assessment with Learned Alignment for Oral Presentations

This notebook follows the implementation structure described in the provided project PDF:

- Visual: OpenFace features
- Acoustic: MFCC, pitch/energy-related features
- Text: BERT/RoBERTa embeddings
- Physiological: HR, EDA, skin temperature
- 1-second temporal alignment
- Modality normalization
- BiLSTM + positional encoding + multi-head self-attention
- 12 directed cross-modal attention interactions
- Learned temporal aggregation
- Missing-modality reconstruction
- Five-task prediction: overall, content, delivery, engagement, anxiety
- Weighted multi-task MSE
- AdamW training + validation + early stopping
- Pearson, Spearman, MAE and RMSE evaluation

> **Important:** This notebook has two layers:
> 1. A fully runnable **dummy-data prototype** for verifying the TEMPAL architecture.
> 2. A **real SOPHIAS adapter section** where actual downloaded/preprocessed SOPHIAS files are connected.
>
> The SOPHIAS raw data access is controlled by the dataset's access/licensing process, so this notebook does not pretend that raw data are automatically available from GitHub.

## 0. Kaggle setup

Recommended Kaggle configuration:

**Notebook → Settings → Accelerator → GPU**

The model uses PyTorch. GPU is helpful for the BERT/text stage and TEMPAL training.

The official project implementation lists packages such as PyTorch, Transformers, librosa, Whisper and datasets. In practice, Whisper is installed as `openai-whisper`, while OpenFace is normally used as an external feature-extraction tool rather than relying on a simple Kaggle `pip install openface` workflow.

In [ ]:
# Check Python and GPU
import sys, os, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("PyTorch check failed:", e)

In [ ]:
# Install the main Python dependencies.
# Run this cell once near the beginning of the Kaggle notebook.

!pip -q install -U librosa transformers datasets scikit-learn scipy pandas numpy matplotlib seaborn
!pip -q install -U openai-whisper

## 1. Imports and configuration

The project PDF specifies the main hyperparameters as:

- hidden dimension = 256
- attention heads = 8
- dropout = 0.2
- learning rate = 1e-4
- batch size = 32
- maximum epochs = 50
- early stopping patience = 10
- optimizer = AdamW

The PDF contains a 4-head mention in one methodology subsection, while the later hyperparameter/code sections use 8 heads. This notebook follows the later implementation configuration: **8 heads**.

In [ ]:
import os
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "hidden_dim": 256,
    "num_heads": 8,
    "dropout": 0.2,
    "lr": 1e-4,
    "batch_size": 32,
    "max_epochs": 50,
    "patience": 10,
    "num_tasks": 5,
    "time_window_seconds": 1.0,
}

print("Device:", DEVICE)
print(CONFIG)

## 2. TEMPAL input/output definition

For each presentation:

`X = {Xv, Xa, Xt, Xp}`

where:

- `Xv`: visual features
- `Xa`: acoustic features
- `Xt`: text features
- `Xp`: physiological features

All modalities are aligned to a common **1-second temporal grid**.

The output contains five scores in the 1–5 range:

1. overall
2. content
3. delivery
4. engagement
5. anxiety

In [ ]:
# Expected modality dimensions from the project description.
# These are configurable because actual exported feature files may differ.

DEFAULT_DIMS = {
    "visual": 50,
    "acoustic": 30,
    "text": 768,
    "physio": 10,
}

TASK_NAMES = [
    "overall",
    "content",
    "delivery",
    "engagement",
    "anxiety",
]

LOSS_WEIGHTS = torch.tensor([1.0, 0.8, 0.8, 0.6, 0.5], dtype=torch.float32)

print("Default input dimensions:", DEFAULT_DIMS)
print("Tasks:", TASK_NAMES)

## 3. Optional: inspect the Kaggle input directory

If you upload your SOPHIAS-derived feature files as a Kaggle Dataset, inspect the exact structure first.

Do **not** assume a folder structure from the paper. The adapter later in this notebook is intentionally generic.

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")

if KAGGLE_INPUT.exists():
    for p in list(KAGGLE_INPUT.iterdir())[:50]:
        print(p)
else:
    print("Kaggle input directory not found in this environment.")

## 4. Feature extraction utilities

### 4.1 Acoustic features

The project methodology mentions:

- MFCC
- pitch/F0
- energy
- speech rate
- pause duration

The function below provides a practical frame-level acoustic extraction base using librosa. The exact final acoustic dimension should be kept consistent with the actual feature schema used in the experiment.

In [ ]:
import librosa

def extract_acoustic_features(audio_path, sr=16000, n_mfcc=13):
    y, sr = librosa.load(audio_path, sr=sr)

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)

    # Energy proxy
    rms = librosa.feature.rms(y=y)

    # Zero crossing rate
    zcr = librosa.feature.zero_crossing_rate(y)

    # Spectral features can help expand the acoustic representation.
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)

    # Align all frame-level features to the shortest number of frames.
    n = min(mfcc.shape[1], rms.shape[1], zcr.shape[1], centroid.shape[1])

    feat = np.concatenate(
        [
            mfcc[:, :n],
            rms[:, :n],
            zcr[:, :n],
            centroid[:, :n],
        ],
        axis=0,
    ).T

    return feat.astype(np.float32)

# Example:
# acoustic = extract_acoustic_features("/kaggle/input/.../audio.wav")
# print(acoustic.shape)

### 4.2 Whisper transcription

Whisper is used for speech transcription. A proper temporal text pipeline should preserve timestamps so that transcript segments can be aligned to the 1-second grid.

For a first prototype, transcription can be generated before constructing timestamp-aligned text embeddings.

In [ ]:
# Whisper installation is handled above.
# Uncomment when you have an audio file.

# import whisper
# whisper_model = whisper.load_model("base")
# result = whisper_model.transcribe("/kaggle/input/.../audio.wav")
# print(result["text"])
# print(result.get("segments", [])[:2])

### 4.3 BERT text embeddings

The project uses BERT/RoBERTa-style embeddings with a target representation of approximately 768 dimensions.

For a real experiment, use timestamped transcript segments and/or slide timing so each 1-second window receives the appropriate text representation. Repeating one global embedding over all time steps is only a prototype baseline.

In [ ]:
from transformers import AutoTokenizer, AutoModel

TEXT_MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE)
text_model.eval()

@torch.no_grad()
def bert_embedding(text):
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256,
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
    output = text_model(**encoded)

    # Mean pooling over valid tokens.
    mask = encoded["attention_mask"].unsqueeze(-1)
    pooled = (output.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return pooled.squeeze(0).cpu().numpy().astype(np.float32)

# Example:
# emb = bert_embedding("This is an example presentation sentence.")
# print(emb.shape)

### 4.4 Physiological features

The project description includes HR, EDA and skin temperature, with statistical/derivative features.

Because the exact SOPHIAS exported column names should be checked against the actual files, this helper uses configurable column names.

In [ ]:
def physiological_features_from_dataframe(
    df,
    hr_col="heart_rate",
    eda_col="eda",
    temp_col="skin_temp",
):
    required = [hr_col, eda_col, temp_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing physiological columns: {missing}")

    hr = pd.to_numeric(df[hr_col], errors="coerce").interpolate().bfill().ffill().to_numpy()
    eda = pd.to_numeric(df[eda_col], errors="coerce").interpolate().bfill().ffill().to_numpy()
    temp = pd.to_numeric(df[temp_col], errors="coerce").interpolate().bfill().ffill().to_numpy()

    def derivative(x):
        return np.gradient(x).astype(np.float32)

    features = np.column_stack([
        hr,
        eda,
        temp,
        derivative(hr),
        derivative(eda),
        derivative(temp),
        hr ** 2,
        eda ** 2,
        temp ** 2,
        np.ones_like(hr),
    ])

    return features.astype(np.float32)

### 4.5 OpenFace visual features

OpenFace is intended to provide facial action units, gaze and head-pose related features.

The exact OpenFace CSV schema can vary by version/export settings. Therefore, the function below selects available columns from a candidate list rather than inventing missing columns.

For the final paper experiment, record exactly which OpenFace columns were used.

In [ ]:
def load_openface_features(csv_path):
    df = pd.read_csv(csv_path)

    candidates = [
        # Common AU intensity columns
        *[f"AU{i:02d}_r" for i in range(1, 18)],
        # Gaze
        "gaze_0_x", "gaze_0_y", "gaze_1_x", "gaze_1_y",
        # Head pose
        "pose_Rx", "pose_Ry", "pose_Rz",
        "pose_Tx", "pose_Ty", "pose_Tz",
    ]

    available = [c for c in candidates if c in df.columns]

    if not available:
        raise ValueError(
            "No expected OpenFace columns were found. "
            "Inspect df.columns and update the feature mapping."
        )

    features = df[available].apply(pd.to_numeric, errors="coerce")
    features = features.interpolate().bfill().ffill().fillna(0.0)

    return features.to_numpy(dtype=np.float32), available

# Example:
# visual, visual_columns = load_openface_features("/kaggle/input/.../openface.csv")
# print(visual.shape, visual_columns)

## 5. One-second temporal alignment

TEMPAL assumes that every modality is mapped to the same 1-second time grid.

For continuous features, a practical implementation is mean pooling within each second.

This function converts timestamped frame-level features into `[T, D]`.

In [ ]:
def mean_pool_to_1s(features, timestamps, duration=None):
    features = np.asarray(features, dtype=np.float32)
    timestamps = np.asarray(timestamps, dtype=np.float32)

    if features.ndim != 2:
        raise ValueError("features must have shape [N, D]")
    if len(timestamps) != len(features):
        raise ValueError("timestamps and features must have the same length")

    if duration is None:
        duration = float(np.nanmax(timestamps)) + 1.0

    T = max(1, int(math.ceil(duration)))
    D = features.shape[1]

    aligned = np.zeros((T, D), dtype=np.float32)

    for t in range(T):
        mask = (timestamps >= t) & (timestamps < t + 1.0)
        if mask.any():
            aligned[t] = np.nanmean(features[mask], axis=0)
        elif t > 0:
            aligned[t] = aligned[t - 1]

    return aligned

# Example:
# aligned_visual = mean_pool_to_1s(visual, openface_timestamps)
# print(aligned_visual.shape)

## 6. Normalization

The methodology specifies zero-mean/unit-variance normalization for each modality.

**Important:** fit normalization statistics on the training participants only, then apply the same statistics to validation/test data. This prevents data leakage.

In [ ]:
class FeatureNormalizer:
    def __init__(self):
        self.scaler = StandardScaler()

    def fit(self, x):
        x = np.asarray(x)
        flat = x.reshape(-1, x.shape[-1])
        self.scaler.fit(flat)
        return self

    def transform(self, x):
        x = np.asarray(x)
        shape = x.shape
        flat = x.reshape(-1, shape[-1])
        out = self.scaler.transform(flat)
        return out.reshape(shape).astype(np.float32)

    def fit_transform(self, x):
        return self.fit(x).transform(x)

## 7. Positional encoding

The TEMPAL methodology specifies positional information before temporal self-attention.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)

        # Handles odd hidden dimensions safely.
        odd_width = pe[:, 1::2].shape[1]
        pe[:, 1::2] = torch.cos(position * div_term[:odd_width])

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        # x: [B, T, D]
        return x + self.pe[:, :x.size(1)]

## 8. Temporal encoder

Each modality uses:

1. Linear projection to the shared hidden dimension
2. BiLSTM
3. Positional encoding
4. Multi-head self-attention
5. LayerNorm
6. Dropout

This produces a temporal representation for each modality.

In [ ]:
class TemporalEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_heads=8, dropout=0.2):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, hidden_dim)

        self.bilstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        self.positional_encoding = SinusoidalPositionalEncoding(hidden_dim)

        self.self_attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        x = self.input_proj(x)
        x, _ = self.bilstm(x)
        x = self.positional_encoding(x)

        attn_out, _ = self.self_attn(
            x, x, x,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )

        x = self.norm(x + self.dropout(attn_out))
        return x

## 9. Directed cross-modal attention

There are `4 × 3 = 12` directed interactions:

- visual ← acoustic
- visual ← text
- visual ← physiological
- acoustic ← visual
- acoustic ← text
- acoustic ← physiological
- text ← visual
- text ← acoustic
- text ← physiological
- physiological ← visual
- physiological ← acoustic
- physiological ← text

The direction matters because the query modality is different from the source modality.

In [ ]:
class CrossModalAttention(nn.Module):
    def __init__(self, hidden_dim=256, num_heads=8, dropout=0.2):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, source, source_key_padding_mask=None):
        out, weights = self.attn(
            query=query,
            key=source,
            value=source,
            key_padding_mask=source_key_padding_mask,
            need_weights=True,
        )

        return self.norm(query + self.dropout(out)), weights

## 10. Learned temporal aggregation

Instead of simple mean pooling, TEMPAL learns a scalar attention score for each time step and uses a weighted sum.

In [ ]:
class LearnedAggregation(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.score = nn.Linear(hidden_dim, 1)

    def forward(self, x, mask=None):
        # x: [B, T, D]
        logits = self.score(x).squeeze(-1)

        if mask is not None:
            logits = logits.masked_fill(mask, -1e9)

        weights = torch.softmax(logits, dim=1)
        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)

        return pooled, weights

## 11. TEMPAL model

This implementation follows the architecture described in the project while making the cross-modal outputs actually participate in the reconstruction/fusion path.

For each target modality:

- its own temporal representation is available;
- incoming cross-modal representations are computed from the other modalities;
- available incoming representations are averaged to form a reconstructed target representation;
- if no source is available, the original target representation is retained.

The final fused vector is passed through a shared 128-unit layer and five task-specific output heads.

In [ ]:
class TEMPAL(nn.Module):
    def __init__(
        self,
        visual_dim=50,
        acoustic_dim=30,
        text_dim=768,
        physio_dim=10,
        hidden_dim=256,
        num_heads=8,
        dropout=0.2,
    ):
        super().__init__()

        self.modalities = ["visual", "acoustic", "text", "physio"]

        self.encoders = nn.ModuleDict({
            "visual": TemporalEncoder(visual_dim, hidden_dim, num_heads, dropout),
            "acoustic": TemporalEncoder(acoustic_dim, hidden_dim, num_heads, dropout),
            "text": TemporalEncoder(text_dim, hidden_dim, num_heads, dropout),
            "physio": TemporalEncoder(physio_dim, hidden_dim, num_heads, dropout),
        })

        self.cross_attn = nn.ModuleDict()

        for target in self.modalities:
            for source in self.modalities:
                if target != source:
                    self.cross_attn[f"{target}_from_{source}"] = CrossModalAttention(
                        hidden_dim, num_heads, dropout
                    )

        self.aggregation = LearnedAggregation(hidden_dim)

        self.shared = nn.Sequential(
            nn.Linear(hidden_dim * 4, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.heads = nn.ModuleDict({
            task: nn.Linear(128, 1)
            for task in TASK_NAMES
        })

    def forward(self, inputs, padding_masks=None, modality_masks=None):
        # inputs: dict of four tensors, each [B, T, D]
        encoded = {}
        for m in self.modalities:
            mask = None if padding_masks is None else padding_masks.get(m)
            encoded[m] = self.encoders[m](inputs[m], key_padding_mask=mask)

        # 12 directed interactions.
        incoming = {m: [] for m in self.modalities}

        for target in self.modalities:
            for source in self.modalities:
                if target == source:
                    continue

                source_mask = None if padding_masks is None else padding_masks.get(source)

                out, _ = self.cross_attn[f"{target}_from_{source}"](
                    encoded[target],
                    encoded[source],
                    source_key_padding_mask=source_mask,
                )

                # A modality-level availability mask can disable this interaction.
                if modality_masks is not None:
                    available = modality_masks.get(source)
                    if available is not None:
                        out = out * available.view(-1, 1, 1).to(out.dtype)

                incoming[target].append(out)

        # Missing-modality reconstruction / cross-modal enrichment.
        reconstructed = {}
        for target in self.modalities:
            own = encoded[target]

            if incoming[target]:
                source_stack = torch.stack(incoming[target], dim=0)
                reconstructed[target] = source_stack.mean(dim=0)
            else:
                reconstructed[target] = own

        # Learned temporal aggregation.
        pooled = {}
        attention_weights = {}

        for m in self.modalities:
            pooled[m], attention_weights[m] = self.aggregation(reconstructed[m])

        fused = torch.cat(
            [pooled["visual"], pooled["acoustic"], pooled["text"], pooled["physio"]],
            dim=-1,
        )

        shared = self.shared(fused)

        outputs = {
            task: self.heads[task](shared).squeeze(-1)
            for task in TASK_NAMES
        }

        return outputs, attention_weights

## 12. Weighted multi-task MSE loss

The project specifies:

`lambda = [1.0, 0.8, 0.8, 0.6, 0.5]`

for:

`[overall, content, delivery, engagement, anxiety]`.

In [ ]:
def weighted_multitask_mse(preds, targets, weights=LOSS_WEIGHTS):
    weights = weights.to(next(iter(preds.values())).device)

    total = 0.0
    task_losses = {}

    for i, task in enumerate(TASK_NAMES):
        loss = F.mse_loss(preds[task], targets[:, i])
        task_losses[task] = loss.detach().item()
        total = total + weights[i] * loss

    return total, task_losses

## 13. Dataset class

Each presentation is represented as:

- visual `[T, Dv]`
- acoustic `[T, Da]`
- text `[T, Dt]`
- physiological `[T, Dp]`
- target `[5]`

The project assumes 1-second aligned sequences. Real presentations may have different durations, so this class pads/truncates to a chosen `max_len`.

In [ ]:
class SOPHIASDataset(Dataset):
    def __init__(self, samples, max_len=60):
        self.samples = samples
        self.max_len = max_len

    def _pad_or_truncate(self, x):
        x = np.asarray(x, dtype=np.float32)
        T, D = x.shape

        if T >= self.max_len:
            return x[:self.max_len], np.zeros(self.max_len, dtype=bool)

        out = np.zeros((self.max_len, D), dtype=np.float32)
        out[:T] = x

        mask = np.ones(self.max_len, dtype=bool)
        mask[:T] = False

        return out, mask

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        visual, vm = self._pad_or_truncate(s["visual"])
        acoustic, am = self._pad_or_truncate(s["acoustic"])
        text, tm = self._pad_or_truncate(s["text"])
        physio, pm = self._pad_or_truncate(s["physio"])

        # Use one shared temporal padding mask for this aligned sample.
        # In a real dataset, all modalities should have the same T after alignment.
        padding_mask = vm

        return {
            "visual": torch.tensor(visual, dtype=torch.float32),
            "acoustic": torch.tensor(acoustic, dtype=torch.float32),
            "text": torch.tensor(text, dtype=torch.float32),
            "physio": torch.tensor(physio, dtype=torch.float32),
            "padding_mask": torch.tensor(padding_mask, dtype=torch.bool),
            "target": torch.tensor(s["target"], dtype=torch.float32),
        }

## 14. Collate function

The model receives a dictionary of modality tensors and a shared temporal padding mask.

In [ ]:
def collate_batch(batch):
    inputs = {
        "visual": torch.stack([b["visual"] for b in batch]),
        "acoustic": torch.stack([b["acoustic"] for b in batch]),
        "text": torch.stack([b["text"] for b in batch]),
        "physio": torch.stack([b["physio"] for b in batch]),
    }

    mask = torch.stack([b["padding_mask"] for b in batch])
    targets = torch.stack([b["target"] for b in batch])

    padding_masks = {m: mask for m in inputs}

    return inputs, padding_masks, targets

# 15. Dummy-data prototype

Before connecting SOPHIAS, run the full model on synthetic data.

This is important because it verifies:

- tensor dimensions
- forward pass
- cross-modal attention
- learned aggregation
- five output heads
- loss calculation
- backpropagation

It does **not** demonstrate real research performance.

In [ ]:
def make_dummy_samples(
    n=100,
    T=60,
    visual_dim=50,
    acoustic_dim=30,
    text_dim=768,
    physio_dim=10,
):
    samples = []

    for _ in range(n):
        samples.append({
            "visual": np.random.randn(T, visual_dim).astype(np.float32),
            "acoustic": np.random.randn(T, acoustic_dim).astype(np.float32),
            "text": np.random.randn(T, text_dim).astype(np.float32),
            "physio": np.random.randn(T, physio_dim).astype(np.float32),
            "target": np.random.uniform(1, 5, size=5).astype(np.float32),
        })

    return samples

dummy_samples = make_dummy_samples()
print("Number of dummy samples:", len(dummy_samples))

In [ ]:
from torch.utils.data import random_split

dummy_dataset = SOPHIASDataset(dummy_samples, max_len=60)

n_total = len(dummy_dataset)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)
n_test = n_total - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    dummy_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    collate_fn=collate_batch,
)

val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=collate_batch,
)

test_loader = DataLoader(
    test_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=collate_batch,
)

print(len(train_ds), len(val_ds), len(test_ds))

## 16. Build TEMPAL

In [ ]:
model = TEMPAL(
    visual_dim=DEFAULT_DIMS["visual"],
    acoustic_dim=DEFAULT_DIMS["acoustic"],
    text_dim=DEFAULT_DIMS["text"],
    physio_dim=DEFAULT_DIMS["physio"],
    hidden_dim=CONFIG["hidden_dim"],
    num_heads=CONFIG["num_heads"],
    dropout=CONFIG["dropout"],
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["lr"],
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3,
)

print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 17. Training and validation

Early stopping follows the project specification: stop when validation performance does not improve for the configured patience.

In [ ]:
def move_inputs_to_device(inputs, padding_masks, targets):
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    padding_masks = {k: v.to(DEVICE) for k, v in padding_masks.items()}
    targets = targets.to(DEVICE)
    return inputs, padding_masks, targets


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    running_loss = 0.0
    n_batches = 0

    for inputs, padding_masks, targets in loader:
        inputs, padding_masks, targets = move_inputs_to_device(
            inputs, padding_masks, targets
        )

        with torch.set_grad_enabled(training):
            preds, _ = model(inputs, padding_masks=padding_masks)
            loss, _ = weighted_multitask_mse(preds, targets)

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        running_loss += loss.item()
        n_batches += 1

    return running_loss / max(n_batches, 1)


def train_model(model, train_loader, val_loader, epochs=50, patience=10):
    history = {"train_loss": [], "val_loss": []}

    best_val = float("inf")
    best_state = None
    wait = 0

    for epoch in range(1, epochs + 1):
        train_loss = run_epoch(model, train_loader, optimizer)
        val_loss = run_epoch(model, val_loader)

        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch:03d} | "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.2e}"
        )

        if val_loss < best_val:
            best_val = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            print("Early stopping.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return history


# For a quick smoke test, use only a few epochs.
# For the project configuration, use CONFIG["max_epochs"].
history = train_model(
    model,
    train_loader,
    val_loader,
    epochs=3,
    patience=2,
)

## 18. Plot training/validation loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Weighted MSE")
plt.title("TEMPAL Training Curve")
plt.legend()
plt.show()

## 19. Evaluation metrics

The project lists:

- Pearson correlation
- Spearman correlation
- MAE
- RMSE

Metrics are reported separately for each of the five tasks.

In [ ]:
def safe_corr(fn, y_true, y_pred):
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan
    try:
        return float(fn(y_true, y_pred)[0])
    except Exception:
        return np.nan


@torch.no_grad()
def predict(model, loader):
    model.eval()

    all_preds = {task: [] for task in TASK_NAMES}
    all_targets = []

    for inputs, padding_masks, targets in loader:
        inputs, padding_masks, targets = move_inputs_to_device(
            inputs, padding_masks, targets
        )

        preds, _ = model(inputs, padding_masks=padding_masks)

        for task in TASK_NAMES:
            all_preds[task].extend(preds[task].detach().cpu().numpy())

        all_targets.append(targets.detach().cpu().numpy())

    all_targets = np.concatenate(all_targets, axis=0)

    results = {}

    for i, task in enumerate(TASK_NAMES):
        y_true = all_targets[:, i]
        y_pred = np.asarray(all_preds[task])

        results[task] = {
            "Pearson": safe_corr(pearsonr, y_true, y_pred),
            "Spearman": safe_corr(spearmanr, y_true, y_pred),
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        }

    return pd.DataFrame(results).T, all_targets, all_preds


metrics_df, y_true, y_pred = predict(model, test_loader)
metrics_df

## 20. Save the trained prototype

The same pattern can be used after training on the actual SOPHIAS-derived features.

In [ ]:
MODEL_DIR = Path("/kaggle/working/TEMPAL_outputs")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "tempal_model.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": CONFIG,
        "input_dims": DEFAULT_DIMS,
        "task_names": TASK_NAMES,
    },
    model_path,
)

print("Saved:", model_path)

# 21. Real SOPHIAS data adapter

## Important

The project PDF does not provide enough information to safely hard-code the exact raw SOPHIAS file names, participant IDs, timestamp column names, or every sensor column name.

Therefore, the correct workflow is:

1. Obtain the authorized SOPHIAS data.
2. Inspect the actual directory/file structure.
3. Identify participant/presentation IDs.
4. Extract each modality.
5. Align each modality to the 1-second grid.
6. Fit normalization using **training participants only**.
7. Build samples.
8. Split by participant identity: 80% train / 10% validation / 10% test.

The following cell helps inspect a mounted SOPHIAS directory.

In [ ]:
def inspect_directory(root, max_files=100):
    root = Path(root)
    print("Root:", root)

    if not root.exists():
        print("Path does not exist.")
        return

    count = 0
    for p in root.rglob("*"):
        if p.is_file():
            print(p)
            count += 1
            if count >= max_files:
                break

# Example:
# inspect_directory("/kaggle/input/your-sophias-dataset")

## 22. Participant-level split

Do not randomly split individual clips from the same participant across train/validation/test.

The project design specifies an **80/10/10 split by participant identity** to prevent participant leakage.

In [ ]:
def participant_split(metadata, participant_col="participant_id", seed=42):
    if participant_col not in metadata.columns:
        raise ValueError(f"Missing column: {participant_col}")

    participants = metadata[participant_col].dropna().unique().tolist()
    rng = np.random.default_rng(seed)
    rng.shuffle(participants)

    n = len(participants)
    n_train = int(0.8 * n)
    n_val = int(0.1 * n)

    train_ids = set(participants[:n_train])
    val_ids = set(participants[n_train:n_train + n_val])
    test_ids = set(participants[n_train + n_val:])

    train = metadata[metadata[participant_col].isin(train_ids)].copy()
    val = metadata[metadata[participant_col].isin(val_ids)].copy()
    test = metadata[metadata[participant_col].isin(test_ids)].copy()

    return train, val, test

# Example:
# train_meta, val_meta, test_meta = participant_split(metadata)

## 23. Example real-data sample builder

Replace the file-loading functions with the actual SOPHIAS-derived files after inspecting their schemas.

Each returned sample must have:

- visual: `[T, Dv]`
- acoustic: `[T, Da]`
- text: `[T, Dt]`
- physio: `[T, Dp]`
- target: `[5]`

The exact target column names should be taken from the actual evaluation/rubric files.

In [ ]:
def build_sample(
    visual_aligned,
    acoustic_aligned,
    text_aligned,
    physio_aligned,
    target_scores,
):
    arrays = [
        np.asarray(visual_aligned, dtype=np.float32),
        np.asarray(acoustic_aligned, dtype=np.float32),
        np.asarray(text_aligned, dtype=np.float32),
        np.asarray(physio_aligned, dtype=np.float32),
    ]

    T_values = [x.shape[0] for x in arrays]

    if len(set(T_values)) != 1:
        raise ValueError(f"Modalities are not temporally aligned: {T_values}")

    target_scores = np.asarray(target_scores, dtype=np.float32)

    if target_scores.shape != (5,):
        raise ValueError(
            f"Expected five scores [overall, content, delivery, engagement, anxiety], "
            f"got shape {target_scores.shape}"
        )

    return {
        "visual": arrays[0],
        "acoustic": arrays[1],
        "text": arrays[2],
        "physio": arrays[3],
        "target": target_scores,
    }

# 24. Recommended real-data pipeline

Use the following order for the actual research run:

**Raw SOPHIAS → modality extraction → timestamp alignment → 1-second pooling → participant split → training-only normalization → Dataset/DataLoader → TEMPAL → validation → test → metrics**

Do not normalize the full dataset before splitting.

Do not use test participants while fitting BERT, scalers, feature statistics, or other learned preprocessing.

For the text stream, preserve timestamps/slide timing whenever available. A single presentation-level text vector repeated across all seconds is a baseline simplification, not a faithful temporal implementation.

# 25. Baseline and ablation experiments

The project proposes the following comparisons:

### Baselines
- Unimodal LSTM
- Simple Fusion
- Late Fusion
- Mean Pooling

### Ablations
- Without cross-modal attention
- Without bidirectional LSTM
- Without learned aggregation
- Without positional encoding
- Without physiological modality

Keep the participant split identical across all experiments so the comparisons remain meaningful.

# 26. Final implementation checklist

Before reporting final results, verify:

- [ ] SOPHIAS access/DUA requirements satisfied
- [ ] Actual file structure inspected
- [ ] Participant IDs identified
- [ ] Visual features extracted from OpenFace
- [ ] Acoustic features extracted
- [ ] Whisper transcription generated where required
- [ ] Timestamped text embeddings generated
- [ ] Physiological features extracted
- [ ] All modalities aligned to 1-second windows
- [ ] Missing values handled
- [ ] Participant-level 80/10/10 split created
- [ ] Normalizer fitted only on training data
- [ ] TEMPAL architecture tested on dummy data
- [ ] TEMPAL trained on real data
- [ ] Best validation checkpoint saved
- [ ] Pearson / Spearman / MAE / RMSE reported
- [ ] Baselines completed
- [ ] Ablation studies completed
- [ ] Expected-result values from the PDF are NOT presented as actual measured results

# 27. Useful project paths

Suggested Kaggle working structure:

```text
/kaggle/working/
└── TEMPAL_outputs/
    ├── tempal_model.pt
    ├── metrics.csv
    ├── training_history.csv
    └── predictions.csv
```

For a larger experiment, also save cached modality features so you do not have to rerun expensive extraction every time.

In [ ]:
# Optional: create output folders
BASE = Path("/kaggle/working/TEMPAL_outputs")
for sub in ["checkpoints", "features", "predictions", "plots", "logs"]:
    (BASE / sub).mkdir(parents=True, exist_ok=True)

print("Created:", BASE)